In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split # this is for splitting the data into training and testing sets
from sklearn.preprocessing import StandardScaler, LabelEncoder # this is for scaling the features and encoding the target variable
import pickle

In [3]:
## Load the dataset
data = pd.read_csv('Churn_Modelling.csv')

data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
## Preprocess the data
# Drop irrelevant features
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1) # axis=1 for columns

data.head()



,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
# here Geography and Gender are categorical features, we need to encode them

# Encode categorical features
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender']) # this will convert Male to 1 and Female to 0



data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
# label_encoder_geography = LabelEncoder()
# data['Geography'] = label_encoder_geography.fit_transform(data['Geography']) # this will convert France to 0, Germany to 1 and Spain to 2

# here we will not use label encoding for Geography because it will create an ordinal relationship between the categories, which is not true. Instead, we will use one-hot encoding.
# data = pd.get_dummies(data, columns=['Geography'], prefix='Geography')

from sklearn.preprocessing import OneHotEncoder
one_hot_encoder_geo = OneHotEncoder() 
geo_encoder = one_hot_encoder_geo.fit_transform(data[['Geography']]) # fit and transform the Geography column
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [7]:
one_hot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [8]:
# to convert the sparse matrix to a dense array and then to a dataframe
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))

In [9]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [ ]:
## combine the original dataframe with the one-hot encoded dataframe with the original data

data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [11]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [12]:
#saving encoders and scalar - one_hot_encoder_geo and label_encoder_gender as pickle file - file in disk, so that we can use this in E2E project

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('one_hot_encoder_geo.pkl','wb') as file:
    pickle.dump(one_hot_encoder_geo,file)

In [13]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [14]:
## divide the dataset into independent and dependent features

x = data.drop('Exited',axis = 1)
y = data['Exited']

## split data in training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

## Scale the features
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.fit_transform(X_test)

In [15]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [16]:
X_test

array([[-5.12501721e-01,  9.09111664e-01, -6.77299309e-01, ...,
        -1.02020406e+00,  1.73668197e+00, -5.63491843e-01],
       [-2.36046598e-01,  9.09111664e-01,  3.84298354e-01, ...,
         9.80196059e-01, -5.75810666e-01, -5.63491843e-01],
       [-4.61306328e-01, -1.09997489e+00,  4.80807232e-01, ...,
        -1.02020406e+00, -5.75810666e-01,  1.77464858e+00],
       ...,
       [ 8.59534812e-01, -1.09997489e+00,  7.70333868e-01, ...,
         9.80196059e-01, -5.75810666e-01, -5.63491843e-01],
       [ 4.70449825e-01,  9.09111664e-01, -9.66825944e-01, ...,
         9.80196059e-01, -5.75810666e-01, -5.63491843e-01],
       [-1.84851205e-01,  9.09111664e-01, -1.73715981e-03, ...,
        -1.02020406e+00,  1.73668197e+00, -5.63491843e-01]])

In [17]:
# saving the scalar of X_train and X_test in pickle format

with open('scalar.pkl','wb') as file:
    pickle.dump(scalar,file)

In [19]:
data

# Now data is ready, on this we can train

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


## ANN Implementation:

We will have input and one or any number of hidden layers. When we take entire ANN (sequential model), where all nodes would be interconnected with weights

Suppose we have an input layer with 2 features and there are two hidden layers, one hidden layer having 3 nodes and another hidden layer has 2 nodes, which are then connected to an output layer with one node.

Number of weights:
i/p to hidden layer1 - 2*3 = 6weights
hidden layer1, having 3 nodes - will have 3 biases = 3weights
hidden layer1 to hidden layer2 = 3*2 = 6 weights
hidden layer2, having 2 nodes - will have 2 biases = 2weights
hidden layer2 to output layer - 2*1 = 2 weights
output bias - 1 weight

Then finally we will apply sigmoid activation function for a binary classification problem statement or a softmax activation function for a multi class classification problem statement

Total number of trainable parameters are = 6+3+6+2+2+1 = 20

So, with help of 20 trainable parameters we will be training in forward and backward propogation.

here we have input dataset from csv file with multiple features

So, the key parameters that we will be specifically using:
When we start creating ANN:

1) Initialize sequential network
2) Whenever we need to create a hidden neuron, we will specifically use dense. for eg. dense = 64 means in that hidden layer there would be 64 neurons
3) Activation function -> sigmoid, tanh, relu or leaky relu. Its always better to use RELU in hidden layer and in output layer use Sigmoid or Softmax
4) Optimizer - useful in backward propogation because these are responsible in updating the weights.
5) Loss function 
6) Metrics - Usually in a classification problem, metrics would be accuracy; in out regression problem, you have MSE, mean squared error, mean absolute error.
7) Training info - store in logs in some folder - so that we can use tensor board - to display the logs in a way that we would be able to understand - Visualization graphs

In [20]:
## ANN Implementation

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


In [24]:
X_train.shape[1],

# this means of a single dimention that have x inputs

(12,)

In [25]:
## Build our ANN model

## give input and hidden layer 1 with 64 neurons
model = Sequential([
    Dense(64, activation='relu',input_shape=(X_train.shape[1],)), # HL1 connected with input layer
    Dense(32, activation='relu'), # HL2
    Dense(1,activation='sigmoid') # output layer, and we used sigmoid because ouput is of form binary classification
]
)

In [26]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate = 0.01) # here its adam optimizer

loss= tensorflow.keras.losses.BinaryCrossentropy()
# 

In [ ]:
## compile the model to do forward and backward propogation

## to compile we use optimizer, loss

model.compile(optimizer=opt,loss="binary_crossentropy",metrics=["accuracy"])

#model.compile(optimizer="adam",loss=loss,metrics=["accuracy"]) 
## this was another way

In [45]:
# Setup the tensorboard so that we can visualize the training logs

from  tensorflow.keras.callbacks import EarlyStopping,TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [36]:
## Setup Early stopping

# For training a neural network, we can use any number epochs.
# Suppose we have given high number of epochs like some 50 epochs, but by 10 epochs, our model got trained at best level and 
# post that the variance of loss would be around 1% to 2%, that is loss difference is not decreasing much, then we can say not to go till
# that many number of epochs, stop early

early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True) # validation loss and patience = 5 means wait for atleast 5 epochs 
# before taking any decision. restore_best_weights - when marked as True means, when doing forward and backward propogation, at which epoch
# you find best weight, consider that when early stopping



In [46]:
### Training the model

history = model.fit(
    X_train,y_train,validation_data = (X_test,y_test),epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3317 - accuracy: 0.8645 - val_loss: 0.3461 - val_accuracy: 0.8575
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3314 - accuracy: 0.8637 - val_loss: 0.3388 - val_accuracy: 0.8625
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3281 - accuracy: 0.8661 - val_loss: 0.3442 - val_accuracy: 0.8590
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3237 - accuracy: 0.8680 - val_loss: 0.3462 - val_accuracy: 0.8610
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3240 - accuracy: 0.8675 - val_loss: 0.3395 - val_accuracy: 0.8630
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3229 - accuracy: 0.8674 - val_loss: 0.3637 - val_accuracy: 0.8570
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3202 - accuracy: 0.8675 - val_loss: 0.3473 - val_accuracy: 0.8615

In [47]:
model.save('model.h5')

c:\Users\bharg\myPython\secondSampleProj\myenv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [48]:
## Load tensorboard extension

%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [49]:

%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 22252), started 0:09:25 ago. (Use '!kill 22252' to kill it.)